ベース：なし（GeoTIFF変換処理）

人口ラスタ：WorldPop "Syrian Arab Republic 100m Population 2026"
https://hub.worldpop.org/geodata/summary?id=75632
syr_worldpop_2026.tif

分析手法：Rasterio Reprojection（EPSG:4326 → EPSG:3857）

対象地域：
・Syria (National Scale)

その他：
・元ラスタ（EPSG:4326）のCRS・サイズ・範囲を確認
・calculate_default_transform() により、EPSG:3857用の新しい座標情報を生成
・reproject() により、人口ラスタをWeb Mercator（EPSG:3857）へ再投影
・transform・width・height を再計算し、新しいGeoTIFFとして保存
・FoliumやWeb地図で利用しやすい3857規格の人口ラスタを作成

In [1]:
# ETL Extract
# 元ラスタ（EPSG:4326）の情報を確認する

import rasterio

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:

    # Check
    # 元ラスタのCRSを確認する
    print(src.crs)

    # Check
    # ラスタの行数・列数を確認する
    print(src.width)
    print(src.height)

    # Check
    # ラスタの地理的範囲（Bounds）を確認する
    print(src.bounds)

EPSG:4326
7992
6011
BoundingBox(left=35.7166658038, bottom=32.310833540089995, right=42.37666577716, top=37.320000186719994)


In [2]:
# ETL Transform
# EPSG:3857へ再投影するための新しい座標情報を計算する

from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling

# 変換先のCRSを指定する
dst_crs = "EPSG:3857"

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:

    # EPSG:3857用のTransform・幅・高さを計算する
    transform, width, height = calculate_default_transform(
        src.crs,
        dst_crs,
        src.width,
        src.height,
        *src.bounds
    )

    # Check
    # 新しいTransform・幅・高さを確認する
    print(transform)
    print(width)
    print(height)

| 100.57, 0.00, 3975961.05|
| 0.00,-100.57, 4483805.04|
| 0.00, 0.00, 1.00|
7372
6757


In [3]:
# ETL Transform / Load
# 人口ラスタをEPSG:3857へ再投影し、新しいGeoTIFFとして保存する

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:

    # 元ラスタのメタデータをコピーする
    kwargs = src.meta.copy()

    # EPSG:3857用にCRS・Transform・幅・高さを更新する
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    # 新しいGeoTIFFファイルを作成する
    with rasterio.open("tif_3_syria_worldpop_3857.tif", "w", **kwargs) as dst:

        # 各バンドをEPSG:3857へ再投影して書き込む
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest
            )